# 01. EDA
데이터 구조 확인 → split 방식(time/random, ratio)과 모델 선택 근거를 정하기 위한 노트북

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from src import data

train = data.load_train()
test = pd.read_csv(data.RAW / "test.csv")
print(train.shape, test.shape)
train.head()

## 1. 기본 규모와 column
time column 유무가 split 방식(time/random)을 결정함

In [ ]:
print("columns:", list(train.columns))
print("time column:", data.find_time_col(train))
print("users:", train.user_id.nunique(), "items:", train.item_id.nunique(), "interactions:", len(train))
print("duplicate (user,item):", train.duplicated(['user_id','item_id']).sum())
density = len(train) / (train.user_id.nunique() * train.item_id.nunique())
print(f"density: {density:.6f}")

## 2. test user와 train user의 관계
train에 없는 test user(cold user)가 많으면 인기순 fallback 비중이 커짐

In [ ]:
tr_users = set(train.user_id)
cold = (~test.user_id.isin(tr_users)).sum()
print(f"test users: {len(test)}, train에 없는 test user: {cold}")
print(f"train user 중 test에 있는 비율: {len(set(test.user_id) & tr_users) / len(tr_users):.3f}")

## 3. user / item interaction 분포
user당 interaction 수 → split의 ratio, min_items 결정. item 수 → EASE max_items 결정

In [ ]:
u_cnt = train.groupby('user_id').size()
i_cnt = train.groupby('item_id').size()
print("user당 interaction:\n", u_cnt.describe(percentiles=[.25,.5,.75,.9,.99]))
print("\nitem당 interaction:\n", i_cnt.describe(percentiles=[.25,.5,.75,.9,.99]))
top = i_cnt.sort_values(ascending=False)
for n in [100, 1000, 10000]:
    print(f"상위 {n} item이 차지하는 interaction 비율: {top.iloc[:n].sum() / len(train):.3f}")

## 4. 시간 분포 (time column이 있을 때)

In [ ]:
tc = data.find_time_col(train)
if tc:
    t = train[tc]
    t = pd.to_datetime(t, unit='s') if pd.api.types.is_numeric_dtype(t) else pd.to_datetime(t)
    print(t.min(), '~', t.max())
    print(t.dt.year.value_counts().sort_index())

## 5. 메타데이터 / 리뷰

In [ ]:
def head_jsonl(path, n=3):
    with open(path) as f:
        return [json.loads(next(f)) for _ in range(n)]

for row in head_jsonl(data.RAW / "item_meta.jsonl"):
    print(json.dumps(row, ensure_ascii=False)[:500], "\n")

In [ ]:
for row in head_jsonl(data.RAW / "train_reviews.jsonl"):
    print(json.dumps(row, ensure_ascii=False)[:500], "\n")